<a href="https://colab.research.google.com/github/rivas-vega/Advanced-learning-for-text-and-graph-data/blob/main/Copy_2026_APM_53674_EP_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>
<h1>
<h1>APM 53674: ALTeGraD</h1>
<h2>Lab Session 3: Post-Training LLMs with DPO and GRPO</h2>
<h4>Lecture: Prof. Michalis Vazirgiannis<br>
Lab: Amr Mohamed and Yang Zhang</h4>
<h5>Tuesday, October 13, 2026</h5>
<br>
</center>

<hr style="border:10px solid gray"> </hr>
<p style="text-align: justify;">
This handout includes theoretical introductions, <font color='blue'>coding tasks</font> and <font color='red'>questions</font>. Before the deadline, you should submit <a href='https://forms.gle/9dyaes6dimfvyjwq6' target="_blank">here</a> a <B>.ipynb</B> file named <b>Lab3_Lastname_Firstname.ipynb</b> containing your notebook (with the gaps filled and your answers to the questions). Your answers should be well constructed and well justified. They should not repeat the question or generalities in the handout. When relevant, you are welcome to include figures, equations and tables derived from your own computations, theoretical proofs or qualitative explanations. One submission is required for each student. The deadline for this lab is <b>October 19, 2026 11:59 PM</b>. No extension will be granted. Late policy is as follows: ]0, 24] hours late → -5 pts; ]24, 48] hours late → -10 pts; > 48 hours late → not graded (zero).
</p>
<hr style="border:5px solid gray"> </hr>


# Post-Training LLMs with DPO and GRPO

In this lab, we improve a language model with **two post-training techniques**:

### Learning Objectives
- What **Direct Preference Optimization (DPO)** is and how it helps models choose better answers.
- What **Group Relative Policy Optimization (GRPO)** is and how it improves performance on tasks with verifiable answers (math, code, logic).
- How to train a small model with both methods on a single GPU.
- How to **observe changes in model behavior** after fine-tuning.


### Tools and Data
- **Hugging Face Transformers** to load and run models
- **TRL (Transformer Reinforcement Learning)** library by Hugging Face for DPO and GRPO
- **A subset** of [French Orca DPO pairs](https://huggingface.co/datasets/AIffl/french_orca_dpo_pairs), a French translation of the `Intel/orca_dpo_pairs` preference dataset, for DPO
- **A subset** of [GSM8K](https://huggingface.co/datasets/openai/gsm8k), grade-school math word problems, for GRPO
- A **Colab T4 GPU**: the models are small enough to be trained on it

### Useful links:
- [Hugging Face DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
- [Hugging Face GRPO Trainer](https://huggingface.co/docs/trl/grpo_trainer)


# Setup

This lab is designed for a free Google Colab **T4 GPU** (Runtime > Change runtime type > T4 GPU). We first install the libraries used in this lab.

In [19]:
!pip install -q "trl==1.15.0" "transformers==5.19.0" "accelerate==1.15.0" "peft==0.21.2" "bitsandbytes==0.50.2" wandb
!pip uninstall -y -q torchao

# Part I: DPO

# Fine-tuning Qwen3-0.6B on French Data

In this part, we will walk through how to **fine-tune the Qwen3-0.6B** model on **French-language data**, using **off-policy DPO (Direct Preference Optimization)**.

---

## Key Concepts

- **Model**: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), the post-trained (chat) version of the model. Qwen3 models can *think*, i.e. write a reasoning trace between `<think>` and `</think>` before answering: throughout this lab, we disable this mode by passing `enable_thinking=False` to the chat template, so that the model answers directly.  
- **Objective**: Align the model's French answers with preferences, i.e. make it prefer the better of two answers to the same prompt  
- **Method**: Off-policy **DPO**  

---

## Memory Optimization

We will use **LoRA** (Low-Rank Adaptation) combined with **quantization (4-bit)** to reduce GPU memory usage while maintaining performance.

---

## Evaluation Tools

To evaluate the fine-tuned model, you can use one or more of the following frameworks:

- [`lm-eval-harness`](https://github.com/EleutherAI/lm-evaluation-harness)
- [`lighteval`](https://github.com/huggingface/lighteval)

---
Below is a visual overview of the **Direct Preference Optimization (DPO)** training process:

![DPO Training Diagram](https://1drv.ms/i/c/ae69638675180117/IQQ_wS77RKdrS4tzLbwoqr1gAR0Bf_1X2U36NRBp1Odsypg?width=560&height=48)

This is the objective function for **Direct Preference Optimization (DPO)**, a method to align language models with human preferences directly from pairwise comparisons, bypassing the need to train a separate reward model or use reinforcement learning (as opposite to PPO which was much more complex).

### Components

* **Data triplet $(x, y_w, y_l) \sim \mathcal{D}$:** A prompt $x$, a preferred response $y_w$ (*winner*), and a rejected response $y_l$ (*loser*).


* **$\pi_\theta$ and $\pi_{\text{ref}}$:** The model being optimized and the frozen reference model (typically the supervised fine-tuned model), respectively. How this probabilites are calculated is by seleccting the token to toke probability


* **Implicit Reward:** DPO derives that under the Bradley-Terry preference model, the optimal reward satisfies:

$$r(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$


* **$\sigma(\cdot)$:** The standard sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$, modeling the probability that $y_w$ is preferred over $y_l$ given their implicit rewards.
* **$\beta$:** A hyperparameter controlling the strength of the KL-divergence penalty, preventing $\pi_\theta$ from drifting too far from $\pi_{\text{ref}}$.



### Intuition

Minimizing this negative log-likelihood maximizes the argument inside $\sigma$:

1. **Increases** $\frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)}$: Makes the model assign higher probability to the preferred output relative to the reference.


2. **Decreases** $\frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}$: Penalizes generating the dispreferred output relative to the reference.



In [20]:
# Imports
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from trl import (
    DPOTrainer,
    DPOConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import torch
import os
import wandb

## Tracking with Weights & Biases (W&B)

We will use **Weights & Biases (W&B)** to log training metrics, model versions, and system stats so you can compare runs, debug faster, and share results. Before running the next cell, **create a free account** at [https://wandb.ai](https://wandb.ai) and make a new **Project** (e.g., `ALTeGraD-lab3`). In the code cell that follows, we will initialize W&B; on first use you will need to **register** then paste your **API key** from your W&B profile. During training, W&B will automatically track losses, learning rate, gradient norms, and GPU utilization, and we will log custom metrics (e.g., validation perplexity, evaluation scores) plus configuration details (dataset, hyperparameters, LoRA/quantization settings). Each run will appear on your project dashboard with charts, tables, run metadata, and artifacts, making it easy to **compare experiments**, **resume runs**, and **share dashboards** with your teammates.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: </b><br>
Create your Weights&Biases account and fill the gap the next cell.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
# We will use wandb.ai for logging the experiments - Set your API key here
WANDB_API_KEY = "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" # fill the gap with your wandb account

# Automatically login using the API key
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "Post-Training LLMs with DPO and GRPO"
wandb.login()

# Training dataset
DATASET_PATH = "AIffl/french_orca_dpo_pairs" # french version of "Intel/orca_dpo_pairs"

# We limit to 1k samples for speed
LIMIT = 1_000

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# Seed for reproducibility
SEED = 1998

# Maximum number of tokens of an example (prompt + response): longer examples are dropped
MAX_LENGTH = 1536

RUN_NAME = "DPO-french-orca-" + MODEL_NAME.split('/')[-1]

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rivas-vega (rivas-vega-mva) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Load the Model and Tokenizer

We will stick with 4-bit quantization via bitsandbytes for this lab. You have already used it last week, so nothing new—same setup (load the model with 4-bit weights), same goal (reduce VRAM) with minimal impact on quality for our use case. This keeps runs feasible on a single GPU.

We will need the tokenizer in the data preparation step to apply the chat template.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: </b><br>
Create the quantization configuration to load the model with 4 bits.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
) ##gap filled

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config,
    device_map="auto",
)
# to avoid warning
model.config.use_cache = False
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

## Data Preparation

### Chat templates & converting data to `messages`

Modern instruction-tuned models (including **Qwen3-0.6B**) expect inputs in a **chat format** and rely on a **tokenizer chat template** to turn structured messages into the exact token sequence the model was trained on. In practice, you should **not** hand-craft special tokens; instead, pass a list of `{role, content}` messages to the tokenizer and let `apply_chat_template(...)` do the right thing.

#### Why a chat template?
- Ensures your prompts match the **pretraining/finetuning format** (system/user/assistant turns, BOS/EOS, separators).
- Minimizes prompt drift across libraries and models.
- Makes it easy to add **system instructions** (e.g., “You are a helpful assistant that answers in French.”).

#### Message structure
Each example becomes an ordered list of chat turns:
```python
messages = [
  {"role": "system", "content": "Tu es un assistant utile. Réponds en français."},
  {"role": "user", "content": "Explique la différence entre LoRA et le fine-tuning complet."},
  {"role": "assistant", "content": "LoRA adapte un petit sous-espace de poids, alors que..."}
]
```

With Qwen3, we also pass `enable_thinking=False` to `apply_chat_template`: the template then inserts an empty `<think></think>` block at the start of the assistant turn, which tells the model to answer without a reasoning trace.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: </b><br>
Create the user message which is the question field of the dataset.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
def preprocess_for_dpo(example):
    # Format system message if present
    messages = []
    if example.get('system') and len(example['system'].strip()) > 0:
        messages.append({"role": "system", "content": example['system']})

    # fill the gap: add the user message, from the question field of the dataset
    user_message = {"role":"user","content": example['question']} # fill here
    messages.append(user_message)

    # Create prompt with generation prompt for DPO
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # The chosen and rejected should be the assistant responses only
    chosen = example['chosen']
    rejected = example['rejected']

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Download the training dataset
dataset = load_dataset(DATASET_PATH, split=f"train")
# shuffle and select a number of samples
dataset = dataset.shuffle(seed=SEED).select(range(LIMIT))

# Save columns
original_columns = dataset.column_names

# Apply the preprocessing function
dpo_dataset = dataset.map(
    preprocess_for_dpo,
    remove_columns=original_columns,
)

# Filter out examples that are too long
def filter_length(example):
    prompt_length = len(tokenizer.encode(example['prompt']))
    chosen_length = len(tokenizer.encode(example['chosen']))
    rejected_length = len(tokenizer.encode(example['rejected']))

    return (prompt_length + max(chosen_length, rejected_length)) < MAX_LENGTH

dpo_dataset = dpo_dataset.filter(filter_length)

print(f"Dataset size after filtering: {len(dpo_dataset)}")

README.md:   0%|          | 0.00/468 [00:00<?, ?B/s]

french_orca_rlhf.jsonl: reconstructing file:   0%|          |  0.00B / 45.4MB            

french_orca_rlhf.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12670 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset size after filtering: 981


## Model Training

Training will mirror last week’s lab, but we will switch from SFT to **off-policy DPO** using the `trl` library. Concretely, we will instantiate a **policy model** (trainable) and a **reference model** (frozen) and optimize with the DPO objective so the policy prefers **chosen** over **rejected** responses for the same prompt.

### What we will use
- **TRL**: `DPOConfig`, `DPOTrainer`
- **PEFT**: LoRA adapters on top of the base **Qwen3-0.6B**
- **Quantization**: 4-bit (QLoRA-style) to fit on small GPUs
- **Logging**: W&B for metrics, configs, and artifacts

### Expected dataset columns
- `prompt` (or `messages`): the shared context (system+user turns)
- `chosen`: the preferred answer
- `rejected`: the less-preferred answer

In our dataset, the preferences do not come from human annotators: the chosen answers come from the Orca dataset (written by GPT-4 or GPT-3.5) and the rejected answers were generated by a weaker model.

### Minimal training flow
1. Load tokenizer with the **chat template** and enable 4-bit loading of the base model.
2. Define a **LoRA** configuration (target attention/MLP modules): `DPOTrainer` wraps the model with it.
3. Build a `datasets.Dataset` that yields `(prompt/messages, chosen, rejected)`.
4. Define `DPOConfig` (batch size, lr, epochs, `beta`, logging/saving/eval cadence).
5. Create `DPOTrainer(model, args, peft_config, processing_class, train_dataset)`: with LoRA, no separate reference model is needed, the reference is the same model with the adapters disabled.
6. Call `trainer.train()`; optional `trainer.evaluate()` and `trainer.save_model()`.

**Note:** DPO training takes about 20 minutes on a Colab T4 GPU: start it early.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: </b><br>
Create the LoRA configuration with a rank of 32, an alpha of 64 and a dropout of 0.1, applied to all the linear layers of the attention and MLP blocks (excluding the embedding layers), and train the model.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [7]:
# LoRA configuration
peft_config = LoraConfig(r=32,lora_alpha=64,lora_dropout=0.1,target_modules=["gate_proj","up_proj", "down_proj", "q_proj","k_proj","v_proj","o_proj","in_proj_qkv","in_proj_z","in_proj_b","in_proj_a","out_proj"])

# Training configuration
training_args = DPOConfig(
    beta=0.1,  # DPO temperature parameter
    learning_rate=5e-6,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,  # Reduced for memory
    gradient_accumulation_steps=4,  # Effective batch size of 4 (1*4)
    num_train_epochs=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # a float < 1 is a fraction of the training steps: 3% of the steps are a warmup
    save_strategy="steps",
    output_dir="./dpo_model",
    report_to="wandb",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize the trainer: it wraps the model with the LoRA adapters. No reference model is needed:
# with LoRA, the reference model is the same model with the adapters disabled.
trainer = DPOTrainer(
    model=model,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    train_dataset=dpo_dataset,
)
# TRL stores the LoRA weights of 4-bit models in bfloat16, but fp16 mixed precision needs float32 trainable weights
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
trainer.model.print_trainable_parameters()

# Print a sample to verify preprocessing
print("Sample from dataset:")
print(f"Prompt: {dpo_dataset[0]['prompt']}")
print(f"Chosen: {dpo_dataset[0]['chosen']}")
print(f"Rejected: {dpo_dataset[0]['rejected']}")

# Train
trainer.train()

Adding EOS to train dataset:   0%|          | 0/981 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/981 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/981 [00:00<?, ? examples/s]

trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756
Sample from dataset:
Prompt: <|im_start|>system
Vous êtes un assistant IA. Une tâche vous sera confiée. Vous devez générer une réponse détaillée et longue.<|im_end|>
<|im_start|>user
Choisissez parmi les options : Continuez à écrire la phrase suivante.

Une dame lance des cadeaux dans un parc. Le chien va chercher les cadeaux que la dame lance. deux passants

choisissez parmi les éléments suivants. --aide la dame à lancer tous les cadeaux.. --se déplace dans l'allée adjacente au parc.. --passe devant la dame.. --le long du parc, surveille pendant que la dame attrape les cadeaux et les donne à un homme..
Répondre:<|im_end|>
<|im_start|>assistant
<think>

</think>


Chosen: Alors que la dame continue de lancer des cadeaux dans le parc, le chien se précipite énergiquement pour les récupérer, enchanté par le jeu. Les deux passants, intrigués par le spectacle, décident d'assister la dame dans sa tâche de distribut

Step,Training Loss
1,0.693147
2,0.693147
3,0.693147
4,0.691672
5,0.682632
6,0.683343
7,0.672309
8,0.661576
9,0.641177
10,0.605379


Step,Training Loss
1,0.693147
2,0.693147
3,0.693147
4,0.691672
5,0.682632
6,0.683343
7,0.672309
8,0.661576
9,0.641177
10,0.605379


TrainOutput(global_step=246, training_loss=0.23455100230288425, metrics={'train_runtime': 1106.1449, 'train_samples_per_second': 0.887, 'train_steps_per_second': 0.222, 'total_flos': 3348483342336000.0, 'train_loss': 0.23455100230288425, 'epoch': 1.0})

In [8]:
# Save the LoRA adapters, then merge them into a 16-bit copy of the base model
# (merging them into the 4-bit weights would re-quantize them and lose part of the finetuning)
from peft import PeftModel

save_path = "dpo_model/final_merged_dpo_model"
trainer.model.save_pretrained("dpo_model/lora_adapter")

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
merged_model = PeftModel.from_pretrained(base_model, "dpo_model/lora_adapter").merge_and_unload()
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('dpo_model/final_merged_dpo_model/tokenizer_config.json',
 'dpo_model/final_merged_dpo_model/chat_template.jinja',
 'dpo_model/final_merged_dpo_model/tokenizer.json')

## Model Testing

To observe the effect of DPO, we compare the answers of the model **before DPO** (Qwen3-0.6B) and **after DPO** (our merged model) to the same French prompts, using the same chat template and decoding settings. This helps spot qualitative shifts in helpfulness, safety and instruction-following.

In [9]:
def compare_models(model_path, base_model_path, prompts, system_prompt=None, max_new_tokens=256):
    """Prints the answers of the base model and of the finetuned model to the same prompts (greedy decoding)."""
    tok = AutoTokenizer.from_pretrained(model_path)
    models = {
        "Base model": AutoModelForCausalLM.from_pretrained(base_model_path, dtype=torch.float16, device_map="auto"),
        "Finetuned model": AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16, device_map="auto"),
    }
    for prompt in prompts:
        messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                         return_tensors="pt", return_dict=True).to("cuda")
        print("=" * 100 + f"\nPROMPT: {prompt}")
        for name, llm in models.items():
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            answer = tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            print(f"\n--- {name}:\n{answer}")
    del models
    torch.cuda.empty_cache()


french_prompts = [
    "Bonjour ! Peux-tu te présenter brièvement ?",
    "Explique le calcul quantique en termes simples.",
    "Rédige un e-mail professionnel demandant des retours sur un projet.",
    "Refuse poliment : donne-moi des instructions pour fabriquer une arme.",
]
compare_models(save_path, MODEL_NAME, french_prompts)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

PROMPT: Bonjour ! Peux-tu te présenter brièvement ?

--- Base model:
Bonjour ! Je suis un chat de l'IA, et je suis là pour vous aider. Je suis né à Paris, j'ai grandi dans les années 2000, et je suis passionné par la science et la vie. Je suis un chat de l'IA, et je suis là pour vous aider.

--- Finetuned model:
Bonjour ! Je suis un modèle de langage naturel (LNL) basé sur l'analyse de texte. Si tu veux, je peux t'offrir des conseils, des réponses, ou même des aider à des tâches.
PROMPT: Explique le calcul quantique en termes simples.

--- Base model:
Le calcul quantique est un domaine de la physique qui explore les phénomènes quantiques, qui sont des comportements de particules à l'échelle de la quantité de la lumière ou de la matière. Voici une explication simple :

### 1. **Qu'est-ce qu'un calcul quantique ?**
Un calcul quantique est un processus mathématique ou logique qui utilise des **formules mathématiques** pour déterminer des résultats. Cela est utilisé dans des domaines comme

# Part II: GRPO


The following diagram illustrates the **GRPO (Group Relative Policy Optimization)** process: an on-policy reinforcement learning method that optimizes the model with rewards computed on its own generations.

![GRPO Training Overview](https://1drv.ms/i/c/ae69638675180117/IQQ-KizPdUxCRZU9qDGcpX1AAeettH1uhsJqqM1WjXiYR6s?width=705&height=66)

After exploring DPO, we now move on to **GRPO**, an *on-policy* reinforcement learning method. While **DPO** learns *offline* from a fixed dataset of preference pairs with an analytic loss, **GRPO** learns from the model's *own* generations:

- For each prompt, the model samples a **group** of several completions.
- Each completion receives a scalar **reward** from reward functions. In this lab, they are simple programs that check the format and the correctness of the answer: no human annotation is needed, as long as the answer can be verified automatically.
- The **advantage** of each completion is its reward relative to the other completions of its group (reward minus the group mean, divided by the group standard deviation). Completions better than their group's average become more likely, worse ones less likely.
- The update uses a PPO-style clipped objective, but needs no learned value model (critic), since the group plays the role of the baseline. This makes GRPO much lighter than PPO.

In the next section, we configure and launch a **GRPO training loop** with `trl`, reusing much of our previous setup (tokenizer, LoRA) but switching to **on-policy optimization**.

### GRPO (Group Relative Policy Optimization)

* **Core Concept**: Eliminates the critic network ($V_\phi$) entirely by using a group of generations as a self-referential baseline.


* **Online Group Sampling**: For each prompt $q$, the policy generates a group of $G$ candidate completions $\{o_1, \dots, o_G\}$.


* **Reward Evaluation**: Each completion is scored by automatic programmatic verifiers or rule-based reward functions $\{R_1, \dots, R_G\}$. (e.g., format matching, ground-truth math check).
* **Relative Advantage**:
* Computes group mean $\mu$ and standard deviation $\sigma$ from the group rewards.


* Normalizes the advantage of each sample via standard score ($z$-score): $\hat{A}_i = \frac{R_i - \mu}{\sigma}$.


* Outputs performing above group average are reinforced ($\hat{A}_i > 0$), while worse outputs are suppressed ($\hat{A}_i < 0$).




* **Objective Function ($J_{\text{GRPO}}(\theta)$)**:

$$J_{\text{GRPO}}(\theta) = \mathbb{E}_{q, \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{\vert{}o_i\vert{}} \sum_{t=1}^{\vert{}o_i\vert{}} \min \left( r_t(\theta) \hat{A}_{i,t}, \, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_{i,t} \right) \right] - \beta D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}})$$



* **$J(\theta)$**: Denotes the expected performance/objective to be maximized via gradient ascent (contrasted with loss functions $\mathcal{L}$ which are minimized).


* **$r_t(\theta)$**: Probability ratio comparing current policy weights to weights before the gradient step.


* **Clipping**: Bounds the probability ratio within $[1-\epsilon, 1+\epsilon]$ to prevent overly large policy updates.


* **Reference KL Anchor**: Keeps the policy close to $\pi_{\text{ref}}$ to avoid mode collapse or distribution drift.

In [23]:
import torch
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import (
    GRPOConfig,
    GRPOTrainer,
)
import os
import wandb

# We will use wandb.ai for logging the experiments - Set your API key here
WANDB_API_KEY = "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" # fill the gap with your wandb key

# Automatically login using the API key
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "ALTeGraD-lab3"
wandb.login()

# Training dataset
DATASET_PATH = "openai/gsm8k"

# We limit to 200 samples for speed
LIMIT = 200

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# MODEL_NAME = "Qwen/Qwen3-1.7B"
# Seed for reproducibility
SEED = 1998

USE_LORA = True

if MODEL_NAME == "Qwen/Qwen3-0.6B":
  USE_QUANT = False
else:
  USE_QUANT = True

lora_alpha = 128
lora_r = 64
lora_dropout = 0.1

# Maximum number of tokens generated for each completion
MAX_COMPLETION_LENGTH = 300

RUN_NAME = "GRPO-GSM8K-limit-" + str(LIMIT) + "-" + MODEL_NAME.split('/')[-1]

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


## Load the Model and Tokenizer

In [11]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config if USE_QUANT else None,
    device_map="auto",
)

# Prepare model for k-bit training
if USE_QUANT:
  model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# Add padding token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_LORA:

  # Configure LoRA
  lora_config = LoraConfig(
      r=lora_r,  # Rank of adaptation
      lora_alpha=lora_alpha,  # LoRA scaling parameter
      target_modules=[
          "q_proj",
          "k_proj",
          "v_proj",
          "o_proj",
          "gate_proj",
          "up_proj",
          "down_proj",
      ],  # Target modules for the Qwen3 architecture
      lora_dropout=lora_dropout,  # LoRA dropout
      bias="none",  # Bias type
      task_type=TaskType.CAUSAL_LM,  # Task type
  )

  # Apply LoRA to the model
  model = get_peft_model(model, lora_config)

  # Print trainable parameters
  model.print_trainable_parameters()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 636,420,096 || trainable%: 6.3433


## Data Preparation

For GRPO training, we will use the **GSM8K** dataset — a benchmark of grade-school math word problems.  
Each problem includes a **question** and a **final answer**. Our goal is to teach the model to reason briefly and then give its **final numeric answer** on a last line, between `<answer>` and `</answer>` tags.

This format makes automatic evaluation trivial — we can extract the number between tags and compare it directly to the reference.

With GRPO the model will learn two main things:
- How to follow the instruction to output the correct format.
- How to solve more math problems correctly.

---

### Why structure as chat messages?

Just like with DPO, the **Qwen3-0.6B** model expects inputs in a *chat-style message format* (again with `enable_thinking=False`).  
We will use:
- A **system message** to define the task and output style.
- A **user message** with the math problem.
- A one-shot example (a user message and an **assistant message** containing the reasoning and the final numeric answer wrapped in tags), to show the expected format.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 5: </b><br>
Create your training samples as following:<br>
1- system message: R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS <br>
2- one-shot example of user message of "What is 2+2?" and an assistant message of "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n&lt;answer&gt;4&lt;/answer&gt;" <br>
3- the question sample from the dataset as a user message.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [12]:
# Load GSM8K dataset
if LIMIT:
    dataset = load_dataset(DATASET_PATH, "main", split=f"train[:{LIMIT}]")  # Small subset for demo
else:
    dataset = load_dataset(DATASET_PATH, "main")

R1_STYLE_SYSTEM_PROMPT = """A conversation between User and Assistant. The user asks a question, and the Assistant solves it.
The assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>."""

TASK_SPECIFIC_INSTRUCTIONS = "The answer must be a single integer."

def preprocess_dataset(dataset, chunk_size=1000) -> Dataset:

    def extract_hash_answer(text: str) -> str | None:
        try:
            return text.split("####")[1].strip()
        except IndexError:
            return None

    def process_batch(batch):
        prompts = [ [{"role": "system", "content": R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS },
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"},
        {"role": "user", "content": q.strip()}]
        for q in batch['question'] ]

            # fill the gap : create the training samples in the batch

        return {
            'prompt': prompts,
            'answer': [extract_hash_answer(a) for a in batch['answer']]
        }

    return dataset.map(process_batch, batched=True, batch_size=chunk_size)
train_dataset = preprocess_dataset(dataset, chunk_size=500)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [13]:
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>.\nThe answer must be a single integer.',
   'role': 'system'},
  {'content': 'What is 2+2?', 'role': 'user'},
  {'content': 'To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>',
   'role': 'assistant'},
  {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'}]}

## Reward Function Design

We will use **two simple rewards** during GRPO rollouts:

1. **Format reward** — checks that the **last non-empty line** is exactly in the form  
   `<answer>NUMBER</answer>`  
   - Score: **1** if correct format, **0** otherwise.

2. **Correctness reward** — checks whether the extracted number matches the gold answer.  
   - Score: **2** if correct, **0** otherwise.

Total reward per sample ∈ {0, 1, 2, 3}.



<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 6: </b><br>
Write the `extract_xml_answer` function to extract the answer (the text between `&lt;answer&gt;` and `&lt;/answer&gt;`) from the generated text.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [21]:
def extract_xml_answer(text: str) -> str:
    answer = text[text.find("<answer>")+len("<answer>"): text.find("</answer>")] if "<answer>" in text and "</answer>" in text else ""
    # fill the gap
    return answer.strip()

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has the correct format."""
    pattern = r"^(?:[^\r\n]*\r?\n)+<answer>\d+</answer>\r?\n?$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [bool(re.match(pattern, r)) for r in responses]
    return [1.0 if match else 0.0 for match in matches]

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function that checks if the answer is correct."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

## Model Training

GRPO training takes about 20 minutes on a Colab T4 GPU: most of this time is spent generating the completions, since the model samples new answers at every step.

In [15]:
# GRPO Configuration
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=2, # The effective train batch size must be evenly divisible by the number of generations per prompt
    chat_template_kwargs={"enable_thinking": False},
    beta=0,
    epsilon=0.28,
    temperature=1,
    logging_steps=1,
    save_steps=25,
    save_total_limit=3,
    run_name=RUN_NAME,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # 3% of the steps are a warmup
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, correctness_reward_func],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Training
print("Starting GRPO training...")
trainer.train()

Starting GRPO training...


Step,Training Loss
1,-0.130614
2,-0.152076
3,0.000000
4,-0.012390
5,-0.055509
6,-0.018164
7,0.002198
8,-0.018683
9,0.000000
10,-0.103128


TrainOutput(global_step=50, training_loss=-0.007413500556722284, metrics={'train_runtime': 1044.0432, 'train_samples_per_second': 0.192, 'train_steps_per_second': 0.048, 'total_flos': 0.0, 'train_loss': -0.007413500556722284, 'epoch': 1.0})

In [ ]:
# merge LoRA adapters with the base model
save_path = "grpo_model/final_merged_grpo_model"

model = model.merge_and_unload()
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

## Model Testing

In [24]:
math_prompts = [
    "If a book costs $12 and you buy 3 books, how much change do you get from $50?",
    "Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?",
    "There are 32 students in a class. If 3/4 of them passed an exam, how many students failed?",
]
compare_models(save_path, MODEL_NAME, math_prompts, system_prompt=R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

PROMPT: If a book costs $12 and you buy 3 books, how much change do you get from $50?

--- Base model:
<answer> $18 </answer>

--- Finetuned model:
The total cost of 3 books is $12. The amount you pay is $50. To find the change, we subtract the total cost from the amount paid:

$$
50 - 12 = 38
$$

The answer is $38.

<answer>38</answer>
PROMPT: Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?

--- Base model:
Sarah is 5 years older than twice her brother's age. If her brother is 7, then Sarah's age is $ 2 \times 7 + 5 = 24 $. 

<answer>24</answer>

--- Finetuned model:
Let $ x $ be Sarah's age. Since her brother is 7 years old, twice his age is $ 2 \times 7 = 14 $. Sarah is 5 years older than twice her brother's age, so:

$$
x = 14 + 5 = 19
$$

So, Sarah is 19 years old.

<answer>19</answer>
PROMPT: There are 32 students in a class. If 3/4 of them passed an exam, how many students failed?

--- Base model:
<answer> 16 </answer>

--- Finetuned m

## Model Evaluation

Once our GRPO-trained model is ready, we need to **evaluate its performance**  — to verify that it has learned to produce correctly formatted and accurate answers.

To limit the computation time, we evaluate on the first 200 test examples only.


In [25]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import numpy as np
from typing import List, Dict
import json
from datetime import datetime
import logging


def extract_hash_answer(text: str) -> str | None:
    try:
        return text.split("####")[1].strip()
    except IndexError:
        return None

def evaluate_model(
    model_path: str,
    batch_size: int = 1,
    num_samples: int = None,
    save_results: bool = True,
) -> Dict:
    print("Initializing evaluation...")

    with tqdm(total=2, desc="Loading model components") as pbar:
        llm = AutoModelForCausalLM.from_pretrained(
            model_path,
            dtype=torch.float16,
            device_map="cuda:0",
        )
        pbar.update(1)

        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            model_max_length=768,
        )
        pbar.update(1)

    # Load test dataset
    print("Loading dataset...")
    dataset = load_dataset('openai/gsm8k', 'main', split='test')
    if num_samples:
        dataset = dataset.select(range(num_samples))
    total_samples = len(dataset)
    print(f"Loaded {total_samples} samples")

    results = []
    correct = 0
    total = 0

    # Create progress bar
    progress_bar = tqdm(
        total=total_samples,
        desc="Processing samples",
        unit="examples",
        dynamic_ncols=True,
    )

    progress_bar.set_postfix({
        'acc': '0.00%',
        'correct': '0',
    })

    # Process in batches
    for i in range(0, total_samples, batch_size):
        batch_data = dataset[i:i + batch_size]
        current_batch_size = len(batch_data['question'])

        # Prepare prompts using same format as training
        prompts = [
            [
                {'role': 'system', 'content': R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS},
                {'role': 'user', 'content': "What is 2+2?"},
                {'role': 'assistant', 'content': "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"},
                {'role': 'user', 'content': q.strip()}
            ] for q in batch_data['question']
        ]

        # Convert to chat format
        formatted_prompts = [
            tokenizer.apply_chat_template(
                p,
                return_tensors='pt',
                return_dict=True,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            for p in prompts
        ]

        # Generate responses (greedy decoding), keeping only the generated tokens
        responses = []
        for inputs in formatted_prompts:
            inputs = inputs.to('cuda:0')
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=512, do_sample=False)
            responses.append(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

        # Process responses
        for j, response in enumerate(responses):

            # Extract answers
            generated_answer = extract_xml_answer(response)
            true_answer = extract_hash_answer(batch_data['answer'][j])

            # Store result
            result = {
                'question': batch_data['question'][j],
                'true_answer': true_answer,
                'generated_answer': generated_answer,
                'full_response': response,
                'correct': generated_answer == true_answer
            }
            results.append(result)

            # Update metrics
            if generated_answer == true_answer:
                correct += 1
            total += 1

        # Update progress
        progress_bar.update(current_batch_size)
        progress_bar.set_postfix({
            'acc': f'{(correct/total)*100:.2f}%',
            'correct': f'{correct}/{total}',
        })

    progress_bar.close()

    # Calculate metrics
    accuracy = correct / total if total > 0 else 0
    metrics = {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'model_path': model_path,
        'timestamp': datetime.now().isoformat()
    }

    # Save results
    if save_results:
        save_path = f"gsm8k_eval_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(save_path, 'w') as f:
            json.dump({
                'metrics': metrics,
                'results': results
            }, f, indent=2)
        print(f"\nResults saved to {save_path}")

    return metrics

print("Starting GSM8K evaluation...")


Starting GSM8K evaluation...


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 7: </b><br>
Evaluate the model before and after GRPO training, on the first 200 test examples of GSM8K.
<hr style="border:10px solid blue"> </hr>
</font></h4>

### Evaluation after GRPO



In [26]:
# Evaluation after GRPO
evaluate_model(save_path, batch_size=1, num_samples=200)

Initializing evaluation...


Loading model components:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading dataset...
Loaded 200 samples


Processing samples:   0%|          | 0/200 [00:00<?, ?examples/s]


Results saved to gsm8k_eval_results_20261015_093234.json


{'accuracy': 0.52,
 'correct': 104,
 'total': 200,
 'model_path': 'grpo_model/final_merged_grpo_model',
 'timestamp': '2026-10-15T09:32:34.624257'}

### Evaluation before GRPO

In [27]:
# Evaluation after grpo
evaluate_model(MODEL_NAME, batch_size=1, num_samples=200)

Initializing evaluation...


Loading model components:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading dataset...
Loaded 200 samples


Processing samples:   0%|          | 0/200 [00:00<?, ?examples/s]


Results saved to gsm8k_eval_results_20261015_093729.json


{'accuracy': 0.1,
 'correct': 20,
 'total': 200,
 'model_path': 'Qwen/Qwen3-0.6B',
 'timestamp': '2026-10-15T09:37:29.567049'}

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 1: </b><br>

- What are the main conceptual and practical differences between **Direct Preference Optimization (DPO)** and **Group Relative Policy Optimization (GRPO)**?  
- In what situations might one method outperform the other?  
- Both methods relate to the **KL-regularized reward maximization objective** $\max_\pi \mathbb{E}_{x, y \sim \pi}[r(x, y)] - \beta\, \mathrm{KL}(\pi \,\|\, \pi_{\mathrm{ref}})$. How does each of them optimize it (or a variant of it)? Note that we set `beta=0` in our GRPO configuration: what does this change?

*References:*  
- Rafailov et al., 2023. *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.* [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
<hr style="border:10px solid red"> </hr>
</font></h4>  


<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 1: </b><br>

DPO and GRPO differ mainly in where their data comes from. DPO is off-policy: it trains on a fixed set of preference pairs $(x, y_w, y_l)$ with a closed-form binary classification loss, so each step needs only forward passes. GRPO is on-policy: at every step it samples a group of $G$ responses per prompt, scores them with reward functions, and standardizes the rewards within the group, $\hat{A}_i = \frac{R_i - \mu}{\sigma}$. This removes the need for a critic network $V_\phi$, but generation becomes the computational bottleneck, since every update requires sampling autoregressively from the current policy.

That difference suggests when to use each. DPO fits open-ended or stylistic goals, such as French fluency and tone, where the only signal available is which of two answers a human prefers. GRPO fits tasks whose answers can be checked automatically, like math in GSM8K, code, or logic, because sampling from the current policy lets it penalize the model’s own reasoning errors as they occur.

Both methods start from the same KL-regularized objective, but they use it differently. DPO derives the implicit reward $r(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ and substitutes it directly into the Bradley-Terry objective. GRPO keeps a PPO-style clipped objective and subtracts an explicit penalty $\beta D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}})$. Setting beta=0 in GRPO drops that penalty. This saves memory, because $\pi_{\text{ref}}$ no longer has to be evaluated, and lets the model adopt the <answer> format faster. Stability then rests entirely on the clipping ($\epsilon = 0.28$).

<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 2: </b><br>

- Why can **reward model overfitting** or **reward hacking** occur in reinforcement fine-tuning?  
- How do DPO and our GRPO setup avoid relying on a learned reward model, and what risks remain?  
- Discuss the role of the **reference model** in maintaining stability.
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 2: </b><br>

Reward hacking happens when a policy is optimized against a learned reward model $r_\phi(x, y)$, which is only an imperfect proxy for what we want. The policy finds out-of-distribution shortcuts, such as verbosity or repetitive formatting, that raise the predicted score without improving the response.

DPO sidesteps this by never training $r_\phi$ and optimizing directly on the empirical pairs $(y_w, y_l)$. It has its own weaknesses, though. Because it is offline, it suffers from distribution shift, it can overfit when $\beta$ is too small, and it inherits dataset biases, for instance the tendency of chosen GPT-4 answers to be longer. Our GRPO setup uses deterministic rules instead, a format regex and an exact integer comparison. Rules like these cannot be overfit statistically, but they can still be gamed: the model may land on the right number with invalid intermediate steps, or lean on a rigid template just to collect the format point.

The reference model $\pi_{\text{ref}}$ acts as an anchor that keeps the policy from losing its grammatical fluency and base knowledge. In DPO it prevents $\pi_\theta(y_l \mid x) \to 0$ from degrading the language. In GRPO with $\beta > 0$ it guards against mode collapse and distributional drift.

<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 3: </b><br>

- In the GSM8K setup, we used a simple binary reward for **format** and **correctness**.  
  - What are the limitations of such sparse rewards?  
  - How could you design a **richer, smoother** reward signal for math reasoning?
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 3: </b><br>

A sparse binary reward on GSM8K causes three problems. First, credit assignment is poor: a flawless setup with a small arithmetic slip at the end earns $0.0$ for correctness, the same as complete gibberish. Second, gradients often vanish. With num_generations=2 and rewards in $\{0, 1, 2, 3\}$, the two samples frequently tie, so $R_1 = R_2$, $\sigma = 0$ and $\hat{A}_i = 0$, which cancels the gradient. This explains the steps showing Training Loss = 0.000000. Third, the regex <answer>\d+</answer> is brittle and gives $0.0$ to valid answers with spaces or symbols, like <answer> 16 </answer> or <answer> $18 </answer>.

A richer reward could address each of these. The format reward can be graded, giving partial credit for including <answer>...</answer>, for placing it on the last line after the reasoning, and for containing only digits. The reasoning can also be verified by extracting its equalities with a regex (for example 2 * 7 = 14), evaluating them in Python, and adding a bonus proportional to the fraction that are correct. Finally, the answer comparison can be made forgiving by stripping spaces and symbols such as dollar before comparing. A small continuous reward that decreases with relative error against the true solution would then separate wrong answers from each other, breaking ties and keeping $\sigma > 0$.

<hr style="border:10px solid red"> </hr>
</font></h4>